# LegaLoom-Env: Full Curriculum GRPO Training

Trains across **all four difficulty levels** in a balanced curriculum (easy → medium → hard → expert), 20 GRPO steps per phase = **80 total steps**. Evaluates with 10 episodes per task for tighter confidence than the 5-episode QuickTrain run.

**Use this notebook if you have ~90 min of T4 (free Colab) or A10G (HF Spaces) compute.**

If you only have ~45 min, use `LegaLoom_QuickTrain.ipynb` instead.

**Outputs (overwrite the existing artifacts in repo root):**
- `reward_curves.png` — phase-aware reward curve
- `before_after.png` — 4-task before/after bar chart
- `training_log.json` — full TRL log_history with phase tags
- `training_scores.json` — baseline + trained means per task

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth 'trl>=0.12.0' openenv-core==0.2.3 fastapi uvicorn pydantic httpx openai pyyaml datasets matplotlib --quiet
print('✓ Installed')

In [ ]:
# Cell 2: Clone repo from HF Space
import subprocess, sys, os
!git clone https://huggingface.co/spaces/aarav0202/legaloom-env /content/legaloom_env 2>&1 | tail -3
sys.path.insert(0, '/content/legaloom_env')
os.chdir('/content/legaloom_env')
print('✓ Cloned')

In [ ]:
# Cell 3: Load model + LoRA (5-8 min)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
print('✓ Model loaded')

In [ ]:
# Cell 4: Measure baseline scores BEFORE any training
# 10 episodes per task for tighter confidence than QuickTrain's 5
from unsloth import FastLanguageModel as FLM
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

FLM.for_inference(model)

print('Measuring baseline (untrained) scores, 10 episodes per task...')
baseline_scores = {}
baseline_per_episode = {}
for task_id in ['task_easy','task_medium','task_hard','task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    baseline_scores[task_id] = sum(scores)/len(scores)
    baseline_per_episode[task_id] = scores
    std = (sum((s-baseline_scores[task_id])**2 for s in scores)/len(scores))**0.5
    print(f'  {task_id}: mean={baseline_scores[task_id]:.3f}  std={std:.3f}  n={len(scores)}')

print(f'\nBaseline avg: {sum(baseline_scores.values())/4:.3f}')

In [ ]:
# Cell 5: Run full curriculum training (60-80 min on T4)
# 4 phases × 20 steps each = 80 total steps across all difficulty levels
from unsloth import FastLanguageModel as FLM
from train_grpo import run_curriculum_training

FLM.for_training(model)

schedule = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
log_history = run_curriculum_training(
    model=model,
    tokenizer=tokenizer,
    task_schedule=schedule,
    steps_per_phase=20,
    examples_per_task=60,
    max_prompt_length=1536,
    max_completion_length=768,
)
print(f'\n✓ Curriculum complete. {len(log_history)} log entries across {len(schedule)} phases.')

In [ ]:
# Cell 6: Plot reward curves with phase boundaries marked
import matplotlib.pyplot as plt
import json

# Save raw log
with open('training_log.json','w') as f:
    json.dump(log_history, f, indent=2, default=str)

# Extract step-indexed reward + phase markers
rewards, phases, phase_tasks = [], [], []
for e in log_history:
    if 'reward' in e:
        rewards.append(e['reward'])
        phases.append(e.get('phase', 0))
        phase_tasks.append(e.get('phase_task_id', '?'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Reward curve with phase shading ---
ax = axes[0]
x = list(range(1, len(rewards)+1))
ax.plot(x, rewards, 'b-', linewidth=1.5, alpha=0.6, label='Episode reward')

# Mark phase boundaries
phase_colors = ['#e8f4ff', '#e8ffe8', '#fff5e8', '#ffe8e8']
phase_starts = []
current_phase = phases[0] if phases else 0
phase_starts.append(0)
for i, p in enumerate(phases):
    if p != current_phase:
        phase_starts.append(i)
        current_phase = p
phase_starts.append(len(rewards))
for i in range(len(phase_starts)-1):
    p_idx = phases[phase_starts[i]] if phase_starts[i] < len(phases) else 0
    ax.axvspan(phase_starts[i]+1, phase_starts[i+1], color=phase_colors[p_idx % 4], alpha=0.4)
    midpoint = (phase_starts[i] + phase_starts[i+1]) / 2 + 1
    task_label = phase_tasks[phase_starts[i]].replace('task_','') if phase_starts[i] < len(phase_tasks) else '?'
    ax.text(midpoint, 0.95, task_label, ha='center', va='top', fontsize=10, fontweight='bold')

# Moving average
if len(rewards) >= 3:
    window = 3
    ma = [sum(rewards[max(0,i-window+1):i+1])/min(i+1,window) for i in range(len(rewards))]
    ax.plot(x, ma, 'r-', linewidth=2.5, label='3-step moving avg')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xlabel('Training Step')
ax.set_ylabel('Episode Reward')
ax.set_title('GRPO Curriculum — Reward across all 4 phases')
ax.set_ylim(0, 1)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# --- Loss curve ---
ax2 = axes[1]
losses = [e['loss'] for e in log_history if 'loss' in e]
if losses:
    ax2.plot(range(1, len(losses)+1), losses, 'g-', linewidth=1.5, alpha=0.8)
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Loss')
ax2.set_title('GRPO Curriculum — Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ reward_curves.png saved')

In [ ]:
# Cell 7: Measure trained scores (10 episodes per task)
FLM.for_inference(model)

print('Measuring trained model scores, 10 episodes per task...')
trained_scores = {}
trained_per_episode = {}
for task_id in ['task_easy','task_medium','task_hard','task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    trained_scores[task_id] = sum(scores)/len(scores)
    trained_per_episode[task_id] = scores
    std = (sum((s-trained_scores[task_id])**2 for s in scores)/len(scores))**0.5
    delta = trained_scores[task_id] - baseline_scores[task_id]
    rel = (delta / baseline_scores[task_id] * 100) if baseline_scores[task_id] > 0 else 0
    print(f'  {task_id}: before={baseline_scores[task_id]:.3f}  after={trained_scores[task_id]:.3f}  std={std:.3f}  Δ={delta:+.3f}  ({rel:+.0f}%)')

print(f'\nBaseline avg: {sum(baseline_scores.values())/4:.3f}')
print(f'Trained avg:  {sum(trained_scores.values())/4:.3f}')
print(f'Lift:         {(sum(trained_scores.values())-sum(baseline_scores.values()))/4:+.3f}')

In [ ]:
# Cell 8: Before/After bar chart
import matplotlib.pyplot as plt
import json

tasks  = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
labels = ['Easy', 'Medium', 'Hard', 'Expert']
before = [baseline_scores[t] for t in tasks]
after  = [trained_scores[t]  for t in tasks]

fig, ax = plt.subplots(figsize=(11, 6))
x = range(len(tasks))
w = 0.36
ax.bar([i-w/2 for i in x], before, w, label='Before GRPO (untrained)', color='#e76f51')
ax.bar([i+w/2 for i in x], after,  w, label=f'After GRPO ({len(set(e.get("phase_task_id") for e in log_history if e.get("phase_task_id")))}-phase curriculum)', color='#2a9d8f')
for i in x:
    ax.text(i-w/2, before[i]+0.015, f'{before[i]:.3f}', ha='center', fontsize=9)
    ax.text(i+w/2, after[i]+0.015,  f'{after[i]:.3f}',  ha='center', fontsize=9)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel('Average Score (10 episodes)')
ax.set_title('LegaLoom-Env: Before vs After Curriculum GRPO Training')
ax.set_ylim(0, 1)
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
plt.show()

# Save scores in same shape as repo's training_scores.json
with open('training_scores.json','w') as f:
    json.dump({
        'baseline': baseline_scores,
        'trained':  trained_scores,
        'baseline_per_episode': baseline_per_episode,
        'trained_per_episode':  trained_per_episode,
        'n_episodes_per_task': 10,
        'training_schedule': ['task_easy','task_medium','task_hard','task_expert'],
        'steps_per_phase': 20,
    }, f, indent=2)
print('✓ before_after.png + training_scores.json saved')

In [ ]:
# Cell 9: Print README replacement table + download all artifacts
from google.colab import files

print('=' * 60)
print('PASTE THIS INTO README.md (replaces the existing table):')
print('=' * 60)
print()
print('| Task | Baseline | After GRPO | Δ |')
print('|------|---------:|-----------:|------:|')
for task_id, label in [('task_easy','`task_easy`'),('task_medium','`task_medium`'),('task_hard','`task_hard`'),('task_expert','`task_expert`')]:
    b = baseline_scores[task_id]
    a = trained_scores[task_id]
    delta_pct = ((a-b)/b*100) if b > 0 else 0
    bold = '**' if a > b else ''
    print(f'| {label} | {b:.3f} | {bold}{a:.3f}{bold} | {delta_pct:+.0f}% |')
avg_b = sum(baseline_scores.values())/4
avg_a = sum(trained_scores.values())/4
avg_delta = ((avg_a-avg_b)/avg_b*100) if avg_b > 0 else 0
print(f'| **Average** | **{avg_b:.3f}** | **{avg_a:.3f}** | **{avg_delta:+.0f}%** |')
print()
print('=' * 60)
print('Downloading artifacts...')
files.download('reward_curves.png')
files.download('before_after.png')
files.download('training_scores.json')
files.download('training_log.json')
print('✓ Drop these 4 files into the repo root, push, done.')